In [1]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import random_split
from model.T_learner import TLearnerEstimator
from data.dataset import load_ihdp_data, IPDHDataset, IPHDDataLoader
from utils.metric import pehe, policy_risk, uplift_curve, uplift_auc_score, qini_curve_industry, qini_auc_score_industry
from utils.plot import plot_uplift_curve, plot_qini_curve

In [2]:
train_path = Path('data/ihdp_npci_1-100.train.npz')
data_dict = load_ihdp_data(train_path)
full_dataset = IPDHDataset(data_dict)
train_ratio = 0.8
train_size = int(train_ratio * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_set, valid_set = random_split(full_dataset, [train_size, valid_size])
train_loader = IPHDDataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = IPHDDataLoader(valid_set, batch_size=32, shuffle=False)

test_path = Path('data/ihdp_npci_1-100.test.npz')
test_data = load_ihdp_data(test_path)
test_set = IPDHDataset(test_data)
test_loader = IPHDDataLoader(test_set, batch_size=32, shuffle=False)

In [3]:
tlearner = TLearnerEstimator(x_dim = 25, hidden_dim = 64, lr=1e-3)
tlearner.fit(train_loader, valid_loader, 100)
tau_hat_list, tau_true_list = tlearner.evaluate(test_loader)

Ep  1 | train=40.4625 | val=35.7130
Ep  2 | train=28.8891 | val=15.4506
Ep  3 | train=7.0115 | val=3.1116
Ep  4 | train=2.4318 | val=1.4751
Ep  5 | train=1.9891 | val=0.9578
Ep  6 | train=1.6414 | val=0.9200
Ep  7 | train=1.4743 | val=0.9430
Ep  8 | train=1.4492 | val=0.9025
Ep  9 | train=1.3703 | val=0.9157
Ep 10 | train=1.3696 | val=0.9333
Ep 11 | train=1.1176 | val=0.9666
Ep 12 | train=1.2013 | val=0.9736
Ep 13 | train=1.1788 | val=0.9843
✅ 早停触发！停止训练
Ep  1 | train=7.8517 | val=5.4415
Ep  2 | train=3.4302 | val=2.1863
Ep  3 | train=2.0124 | val=1.5398
Ep  4 | train=1.5539 | val=1.3160
Ep  5 | train=1.3873 | val=1.2193
Ep  6 | train=1.2875 | val=1.1558
Ep  7 | train=1.1813 | val=1.1129
Ep  8 | train=1.1223 | val=1.0879
Ep  9 | train=1.0571 | val=1.0645
Ep 10 | train=0.9886 | val=1.0588
Ep 11 | train=0.9403 | val=1.1046
Ep 12 | train=0.9086 | val=1.0725
Ep 13 | train=0.8624 | val=1.0550
Ep 14 | train=0.8352 | val=1.0735
Ep 15 | train=0.8165 | val=1.0538
Ep 16 | train=0.8228 | val=1.109

In [5]:
pehe_score = pehe(tau_hat_list, tau_true_list)
print("PEHE:", pehe_score)

PEHE: 0.68955964
